# De-identify clinical free text in Microsoft Fabric / Azure Databricks

This notebook runs the **NoteGuard** pipeline over a lakehouse table so that only
**de-identified** text is written back — *sanitise at source*, inside the Trust's own
governance boundary.

Where it runs:

- **Microsoft Fabric** — attach a Lakehouse, run as-is (`spark` is provided).
- **Azure Databricks** — run as-is against a Unity Catalog / hive table.
- **Plain Jupyter** — no Spark? It falls back to a small pandas sample so you can try it.

The engine is a plain, pip-installable Python package (no services, no callbacks): transparent
rules (checksum-validated NHS numbers, GMC/NMC, postcodes, DOB) unioned with spaCy NER via
Presidio, then redaction or patient-consistent pseudonymisation.

In [ ]:
# One-off per environment. en_core_web_sm keeps the install light; use en_core_web_lg
# for the best name recall (the measured numbers in the repo README use lg).
%pip install "noteguard @ git+https://github.com/yumi-h-1/Automatic-PII-preprocessing-tool" --quiet
!python -m spacy download en_core_web_sm --quiet

## 1. Point at your table

Set the input/output table names and which column holds the free text. Everything else is
untouched — only the text column is de-identified.

In [ ]:
TABLE_IN = "clinical_notes"            # lakehouse / catalog table with free text
TABLE_OUT = "clinical_notes_deid"      # de-identified copy written back
TEXT_COL = "note_text"                 # the free-text column
PATIENT_COL = "person_id"              # optional: keeps pseudonyms patient-consistent
METHOD = "redaction"                   # "redaction" or "pseudonym"

import pandas as pd

try:
    pdf = spark.read.table(TABLE_IN).toPandas()   # Fabric / Databricks
    HAVE_SPARK = True
except NameError:                                  # plain Jupyter — demo sample
    HAVE_SPARK = False
    pdf = pd.DataFrame({
        "person_id": ["p1", "p1", "p2"],
        "note_text": [
            "Pt John Smith, NHS no 943 476 5919, DOB 02/03/1981, lives SW1A 1AA.",
            "Mr Smith reviewed on ward by Dr Lee, GMC 1234567. Plan: discharge.",
            "Jane Doe admitted to Manchester Royal Infirmary, contact 07700 900123.",
        ],
    })

print(f"{len(pdf)} rows loaded (spark={HAVE_SPARK})")

## 2. De-identify

One `Pipeline` and one shared `PseudonymVault` per batch, so the same patient always maps to
the same surrogate across all of their notes. The vault lives only in this session's memory —
it is never written anywhere.

In [ ]:
from src.detect import build_detector
from src.pipeline import Pipeline
from src.transform import PseudonymVault

pipe = Pipeline(build_detector(use_presidio=True), PseudonymVault())

def deidentify(row) -> str:
    pid = str(row[PATIENT_COL]) if PATIENT_COL in row else "batch"
    return pipe.sanitise(str(row[TEXT_COL]), METHOD, pid).sanitised

out = pdf.copy()
out[TEXT_COL] = out.apply(deidentify, axis=1)
out.head()

## 3. Write the de-identified table back

In Fabric/Databricks this lands next to the source table; downstream consumers (reports,
models, extracts) read `TABLE_OUT` and never touch the identifiable original.

In [ ]:
if HAVE_SPARK:
    spark.createDataFrame(out).write.mode("overwrite").saveAsTable(TABLE_OUT)
    print(f"wrote {TABLE_OUT}")
else:
    out  # demo: just show the result

## Notes for information governance

- **Nothing identifiable leaves the workspace** — detection and transformation run inside the
  compute you already govern; there are no external API calls.
- **Pseudonymised data is still personal data** under UK GDPR — treat `TABLE_OUT` accordingly
  until your IG lead signs off the flow. `redaction` mode removes rather than replaces.
- **Measured, not assumed**: the repo publishes a false-negative (missed-identifier) evaluation
  and a live re-check — see the *How safe is it?* tab of the demo app and
  [`docs/NHS_PLATFORMS.md`](https://github.com/yumi-h-1/Automatic-PII-preprocessing-tool/blob/main/docs/NHS_PLATFORMS.md).
- For very large tables, swap the pandas `.apply` for a Spark `pandas_udf` so each executor
  builds its own detector — the pipeline object is self-contained, so this is a mechanical change.